# ScanOps V12-7B → GGUF 변환 (클린, 처음부터 끝까지)

**런타임 = T4 GPU.** 위에서 아래로 셀 ①~⑥ 순서대로 실행만 하면 됨.
처음 한 번 실행 기준으로 짜여있음 (중간 멈췄다 재실행하면 해당 셀만 다시).

이전에 터진 것들 다 처리됨: tensorflow/protobuf 충돌(제거), 컴파일 OOM(프리빌트 바이너리), tar.gz.


## ① 환경 정리 + 의존성 (tensorflow 제거가 핵심)


In [ ]:
# tensorflow 제거(protobuf 충돌) + torchao 제거(peft 버전충돌) — 둘 다 우리가 안 씀
!pip uninstall -y -q tensorflow tensorflow-cpu tf-keras torchao 2>/dev/null
# 필요한 것만 설치 (llama.cpp requirements.txt는 protobuf를 깨서 쓰지 않음)
!pip -q install -U transformers peft accelerate sentencepiece gguf
print('① 의존성 완료')

## ② llama.cpp 변환스크립트 + 프리빌트 양자화 바이너리 (컴파일 안 함)


In [ ]:
!rm -rf llama.cpp && git clone -q https://github.com/ggml-org/llama.cpp
import requests, subprocess
rel = requests.get('https://api.github.com/repos/ggml-org/llama.cpp/releases/latest').json()
asset = next(a for a in rel['assets'] if 'ubuntu-x64' in a['name'] and 'cuda' not in a['name'].lower())
url = asset['browser_download_url']; fname = asset['name']; print('다운로드:', fname)
!wget -q "{url}" -O "{fname}"
!rm -rf llamacpp && mkdir -p llamacpp && tar xzf "{fname}" -C llamacpp
QBIN = subprocess.check_output('find llamacpp -name llama-quantize | head -1', shell=True).decode().strip()
print('llama-quantize:', QBIN if QBIN else '❌ 못 찾음')

## ③ adapter_v12_7b.zip 업로드


In [ ]:
import os, shutil, zipfile
if os.path.exists('adapter7b'): shutil.rmtree('adapter7b')
os.makedirs('adapter7b', exist_ok=True)
from google.colab import files
up = files.upload()   # adapter_v12_7b.zip 선택
z = [k for k in up if k.endswith('.zip')][0]
with zipfile.ZipFile(z) as f: f.extractall('adapter7b')
print('adapter ok:', os.path.exists('adapter7b/adapter_model.safetensors'))

## ④ 베이스 7B + 어댑터 병합 (GPU)


In [ ]:
import os
os.environ['USE_TF'] = '0'; os.environ['USE_FLAX'] = '0'
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
BASE = 'Qwen/Qwen2.5-Coder-7B-Instruct'
tok = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
base = AutoModelForCausalLM.from_pretrained(
    BASE, torch_dtype=torch.float16, device_map='auto',
    low_cpu_mem_usage=True, trust_remote_code=True)
merged = PeftModel.from_pretrained(base, 'adapter7b').merge_and_unload()
merged.save_pretrained('merged7b', safe_serialization=True)
tok.save_pretrained('merged7b')
print('④ merged 저장 완료')

## ⑤ GGUF f16 변환 + Q4_K_M 양자화


In [ ]:
import os
libdir = os.path.dirname(QBIN)
!python llama.cpp/convert_hf_to_gguf.py merged7b --outfile v12_7b.f16.gguf --outtype f16
!LD_LIBRARY_PATH="{libdir}" "{QBIN}" v12_7b.f16.gguf qwen-security-v12-7b.Q4_K_M.gguf Q4_K_M
!rm -f v12_7b.f16.gguf
print('⑤ Q4 크기(GB):', round(os.path.getsize('qwen-security-v12-7b.Q4_K_M.gguf')/1e9, 2))

## ⑥ Q4 GGUF 다운로드 (~4.5GB)
받은 파일을 로컬 `scanops-model/models/`에 넣으면 끝.


In [ ]:
from google.colab import files
files.download('qwen-security-v12-7b.Q4_K_M.gguf')